# Hodge & CIDRE Citation Cartel Detection

This notebook demonstrates **Hodge decomposition-based citation cartel detection** on academic journal citation networks.

## What this artifact does

Citation cartels are groups of journals that artificially inflate each other's impact factors through coordinated citation stacking. This method uses **combinatorial Hodge decomposition** to decompose net citation flows into:

- **Gradient component**: hierarchical flow (prestigious journals receive more than they give)
- **Curl component**: cyclic flow (circular citation rings — the cartel signature)
- **Harmonic component**: global recirculation

Journals with high *curl residual* after removing the gradient (prestige) flow are flagged as potential cartel members.

## 6-Phase Evaluation
1. Real 231-journal network — Hodge AUC on JCR-suppressed stacking journals
2. CIDRE comparison on real data
3. Synthetic n_c=10 network — validates detection on known cartels
4. Clean-base injection study — tests detection under controlled conditions
5. Field-aware vs degree-preserving null model comparison
6. Energy fractions (gradient/curl/harmonic) — real vs synthetic network

**Key finding**: Field-aware null model achieves AUC=0.718 on stacking journals — best signal overall.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab — always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally only to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1',
         'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
import sys
import os
import gc
import json
import math
import time
import resource
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import scipy.sparse as sparse
from scipy.sparse.linalg import lsqr
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading

The `mini_demo_data.json` file contains pre-computed results for 17 representative journals from the real 231-journal network, plus all phase summaries. It loads from GitHub when available, with a local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d509fc-curl-in-the-citation-graph-hodge-flow-de/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} journal examples")
print(f"Network: {meta['n_journals']} journals, {meta['n_stacking_positives']} stacking positives")

## Configuration

These parameters control the synthetic network demo and core algorithms. They are set to **minimum values** that still produce meaningful output. Original paper values are noted in comments.

In [ ]:
# --- Synthetic network parameters ---
N_SYNTH = 100          # number of journals in synthetic network (paper: 800)
N_FIELDS = 4           # number of research fields (paper: 12)
N_CARTELS = 3          # number of injected cartel groups (paper: 10)
CARTEL_SIZE = 3        # journals per cartel, forced to 3 for triangle formation
SYNTH_SEED = 42        # random seed

# --- Hodge decomposition ---
MAX_TRIANGLES = 500_000   # triangle cap for curl computation (paper: 2_000_000)
LSQR_ITER_GRAD = 5000     # max lsqr iterations for gradient (paper: 30000)
LSQR_ITER_CURL = 2000     # max lsqr iterations for curl (paper: 20000)

# --- Bootstrap AUC ---
N_BOOTSTRAP = 500      # bootstrap resamples for CI (paper: 2000)

# --- Null model (Phase 5 demo) ---
N_NULL_SAMPLES = 10    # null permutations per type (paper: 100)

# --- Injection study (Phase 4 demo) ---
K_VALUES = [3, 5]         # cartel sizes to test (paper: [3,4,5,10])
W_FACTORS = [0.5, 1.0, 2.0]  # injection weight multipliers (paper: [0.1,0.3,0.5,1.0,2.0])
N_REPS_INJECTION = 5   # repetitions per condition (paper: 20)

## Core Helpers: Hodge Decomposition

The Hodge decomposition uses the boundary operators of a simplicial complex:
- **B1** (N×E node-edge incidence matrix): encodes the graph topology
- **B2** (E×T edge-triangle incidence matrix): encodes triangles for curl computation

The algorithm solves `min_s ||B1^T s - Y_e||^2` (least-squares for gradient), then projects the residual onto the curl space via B2.

In [ ]:
def build_B1(edges: List[Tuple[int, int]], N: int) -> sparse.csr_matrix:
    """Build N×E node-edge incidence matrix (tail=-1, head=+1)."""
    E = len(edges)
    rows, cols, data = [], [], []
    for e_idx, (i, j) in enumerate(edges):
        rows.extend([i, j])
        cols.extend([e_idx, e_idx])
        data.extend([-1.0, 1.0])
    return sparse.csr_matrix((data, (rows, cols)), shape=(N, E))


def build_edge_list(C: sparse.csr_matrix) -> Tuple[List[Tuple[int, int]], Dict]:
    """Extract sorted oriented edge list (i < j) from sparse matrix."""
    cx = C.tocoo()
    edges_set = set()
    for i, j in zip(cx.row, cx.col):
        if i != j:
            edges_set.add((min(int(i), int(j)), max(int(i), int(j))))
    edges = sorted(edges_set)
    edge_to_idx = {e: k for k, e in enumerate(edges)}
    return edges, edge_to_idx


def compute_Ye(C: sparse.csr_matrix, edges: List[Tuple[int, int]]) -> np.ndarray:
    """Net flow vector Y_e for each oriented edge."""
    Y = C - C.T
    Y_arr = Y.toarray().astype(np.float64)
    return np.array([Y_arr[i, j] for (i, j) in edges], dtype=np.float64)


def enumerate_triangles(edges: List[Tuple[int, int]]) -> List[Tuple[int, int, int]]:
    """Enumerate all triangles in the undirected graph represented by edges."""
    adj = defaultdict(set)
    for (i, j) in edges:
        adj[i].add(j)
        adj[j].add(i)
    triangles = []
    for (i, j) in edges:
        for k in adj[i] & adj[j]:
            if k > j:
                triangles.append((i, j, k))
    return triangles


def build_B2(
    edges: List[Tuple[int, int]],
    edge_to_idx: Dict,
    triangles: List[Tuple[int, int, int]],
    E: int,
) -> Optional[sparse.csr_matrix]:
    """Build E×T curl incidence matrix B2."""
    T = len(triangles)
    if T == 0:
        return None
    rows, cols, data = [], [], []
    for t_idx, (i, j, k) in enumerate(triangles):
        e_ij = edge_to_idx.get((i, j))
        e_jk = edge_to_idx.get((j, k))
        e_ik = edge_to_idx.get((i, k))
        if None in (e_ij, e_jk, e_ik):
            continue
        rows.extend([e_ij, e_jk, e_ik])
        cols.extend([t_idx, t_idx, t_idx])
        data.extend([1.0, 1.0, -1.0])
    return sparse.csr_matrix((data, (rows, cols)), shape=(E, T))


def hodge_pipeline(
    C: sparse.csr_matrix,
    max_triangles: int = MAX_TRIANGLES,
) -> Dict:
    """
    Full Hodge decomposition pipeline.
    Returns: s_star, node_grad_residual, node_curl_raw, triangle_curls,
             energy_fractions, edges, triangles.
    """
    N = C.shape[0]
    edges, edge_to_idx = build_edge_list(C)
    E = len(edges)
    Y_e = compute_Ye(C, edges)

    # Gradient: min_s ||B1^T s - Y_e||^2
    B1 = build_B1(edges, N)
    s_star, _, itn, _, resid = lsqr(B1.T, Y_e, damp=1e-6, atol=1e-10, btol=1e-10, iter_lim=LSQR_ITER_GRAD)[:5]
    Y_grad = B1.T @ s_star
    residual = Y_e - Y_grad

    # Gradient residual per node
    nrs = np.zeros(N)
    nec = np.zeros(N, dtype=int)
    for e_idx, (i, j) in enumerate(edges):
        v = abs(residual[e_idx])
        nrs[i] += v; nrs[j] += v
        nec[i] += 1; nec[j] += 1
    node_grad_residual = nrs / (nec + 1e-10)

    # Triangles
    triangles = enumerate_triangles(edges)
    T = len(triangles)
    logger.info(f"Network: N={N}, E={E}, T={T} triangles, lsqr_itn={itn}")

    # Curl component
    if T > 0 and T < max_triangles:
        B2 = build_B2(edges, edge_to_idx, triangles, E)
        if B2 is not None:
            h_star = lsqr(B2, residual, damp=1e-6, iter_lim=LSQR_ITER_CURL)[0]
            Y_curl_vec = B2 @ h_star
            triangle_curls_raw = (B2.T @ Y_e)  # per-triangle curl (T-vector)
        else:
            Y_curl_vec = np.zeros(E)
            triangle_curls_raw = np.array([])
    else:
        # Direct aggregation fallback
        Y_arr = np.zeros((N, N))
        for e_idx, (i, j) in enumerate(edges):
            Y_arr[i, j] = Y_e[e_idx]
            Y_arr[j, i] = -Y_e[e_idx]
        triangle_curls_raw = np.array([
            Y_arr[i, j] + Y_arr[j, k] - Y_arr[i, k]
            for (i, j, k) in triangles
        ]) if T > 0 else np.array([])
        del Y_arr; gc.collect()
        Y_curl_vec = np.zeros(E)

    Y_harm = residual - Y_curl_vec

    # Energy fractions
    total_E = float(np.dot(Y_e, Y_e))
    if total_E < 1e-15:
        grad_frac, curl_frac, harm_frac = 0.0, 0.0, 0.0
    else:
        grad_frac = float(np.dot(Y_grad, Y_grad) / total_E)
        curl_frac = float(np.dot(Y_curl_vec, Y_curl_vec) / total_E)
        harm_frac = float(np.dot(Y_harm, Y_harm) / total_E)
    logger.info(f"Energy: grad={grad_frac:.3f}, curl={curl_frac:.3f}, harm={harm_frac:.3f} (sum={grad_frac+curl_frac+harm_frac:.3f})")

    # Per-node curl score
    node_curl_sum = np.zeros(N)
    node_tri_count = np.zeros(N, dtype=int)
    if len(triangle_curls_raw) > 0:
        for t_idx, (i, j, k) in enumerate(triangles):
            v = abs(float(triangle_curls_raw[t_idx]))
            for nd in [i, j, k]:
                node_curl_sum[nd] += v
                node_tri_count[nd] += 1
    node_curl_raw = node_curl_sum / (node_tri_count + 1e-10)

    return {
        "s_star": s_star,
        "Y_e": Y_e,
        "Y_grad": Y_grad,
        "residual": residual,
        "Y_curl": Y_curl_vec,
        "Y_harm": Y_harm,
        "node_grad_residual": node_grad_residual,
        "node_curl_raw": node_curl_raw,
        "triangle_curls": triangle_curls_raw,
        "energy_fractions": {"gradient": grad_frac, "curl": curl_frac, "harmonic": harm_frac},
        "edges": edges,
        "edge_to_idx": edge_to_idx,
        "triangles": triangles,
        "N": N, "E": E, "T": T,
    }


def bootstrap_auc(
    scores: np.ndarray,
    labels: np.ndarray,
    B: int = N_BOOTSTRAP,
    seed: int = 42,
) -> Tuple[Optional[float], List]:
    """AUC-ROC with bootstrap 95% CI. Returns (auc, [lo, hi])."""
    scores = np.where(np.isfinite(scores), scores, 0.0)
    if labels.sum() < 2 or (labels == 0).sum() < 2:
        return None, [None, None]
    try:
        auc = float(roc_auc_score(labels, scores))
    except Exception:
        return None, [None, None]
    rng = np.random.RandomState(seed)
    boot = []
    for _ in range(B):
        idx = rng.randint(0, len(labels), len(labels))
        if labels[idx].sum() > 0 and (1 - labels[idx]).sum() > 0:
            try:
                boot.append(roc_auc_score(labels[idx], scores[idx]))
            except Exception:
                pass
    if len(boot) >= 10:
        ci = [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))]
    else:
        ci = [auc, auc]
    return auc, ci

## Synthetic Network Generator

We generate a synthetic citation network with controllable cartel injection. This lets us validate the method when ground truth is known.

- `n_cartels=0` → clean base network (no pre-existing cartels)
- `n_cartels>0` → inject cyclic citation rings (A→B→C→A)

Within-field citations follow a hierarchical prestige model; cross-field citations are sparse.

In [ ]:
def generate_synthetic_network(
    N: int = N_SYNTH,
    n_fields: int = N_FIELDS,
    n_cartels: int = N_CARTELS,
    cartel_size: int = CARTEL_SIZE,
    seed: int = SYNTH_SEED,
) -> Tuple[sparse.csr_matrix, List[Dict], List[Dict]]:
    """
    Generate synthetic citation network.
    n_cartels=0 → clean base with no injected cartels.
    cartel_size is forced to min(cartel_size, 3) to guarantee triangles.
    """
    rng = np.random.RandomState(seed)
    logger.info(f"Generating synthetic: N={N}, fields={n_fields}, cartels={n_cartels}, seed={seed}")

    field_labels = np.repeat(np.arange(n_fields), N // n_fields + 1)[:N]
    prestige = rng.exponential(scale=1.0, size=N)

    C = sparse.lil_matrix((N, N), dtype=np.float64)

    # Within-field citations (dense hierarchical)
    for field in range(n_fields):
        members = np.where(field_labels == field)[0]
        for i in members:
            targets = rng.choice(members, size=min(20, len(members) - 1), replace=False)
            weights = prestige[targets] / (prestige[targets].sum() + 1e-10)
            for t, w in zip(targets, weights):
                if t != i:
                    C[i, t] += max(1, int(rng.poisson(50 * w * prestige[i])))

    # Cross-field citations (sparse)
    for i in range(N):
        n_cross = rng.poisson(3)
        if n_cross > 0:
            targets = rng.choice(N, size=min(n_cross, N - 1), replace=False)
            for t in targets:
                if t != i:
                    C[i, t] += max(1, int(rng.poisson(5 * prestige[t])))

    # Inject cartels (skip if n_cartels=0)
    cartel_nodes_all: List[int] = []
    available = set(range(N))
    all_vals = [v for row in C.data for v in row]
    w_cartel = int(max(all_vals) * 0.6) if all_vals else 100

    for _ in range(n_cartels):
        k_use = min(cartel_size, 3)  # force triangles
        if len(available) < k_use:
            break
        nodes = sorted(rng.choice(list(available), size=k_use, replace=False))
        cartel_nodes_all.extend(nodes)
        available -= set(nodes)
        # Directed ring: A→B→C→A
        for idx in range(k_use):
            u, v = nodes[idx], nodes[(idx + 1) % k_use]
            C[u, v] += w_cartel
            C[v, u] += w_cartel * 0.15

    C_csr = C.tocsr()

    field_names = ["Biology", "Chemistry", "Physics", "Medicine", "Engineering",
                   "Mathematics", "Computer Science", "Environmental Science",
                   "Agriculture", "Materials Science", "Ecology", "Biochemistry"]
    journals = [
        {
            "id": f"https://openalex.org/S{i+1000000:08d}",
            "name": f"Journal_{i:04d}_{field_names[field_labels[i] % len(field_names)]}",
            "issn_l": f"{1000+i//100:04d}-{i%1000:04d}",
            "cited_by_count": int(C_csr[:, i].sum()),
            "synthetic_field": field_names[field_labels[i] % len(field_names)],
        }
        for i in range(N)
    ]
    gt = [
        {
            "openalex_id": journals[node]["id"],
            "name": journals[node]["name"],
            "reason": "citation_stacking",
        }
        for node in cartel_nodes_all
    ]

    logger.info(f"Synthetic: N={N}, nnz={C_csr.nnz}, cartel_nodes={len(cartel_nodes_all)}")
    return C_csr, journals, gt

## Phase 3: Synthetic Network Demo

We generate a small synthetic network with known cartel structure, then run the Hodge pipeline and compute detection AUC. This validates the method where ground truth is available.

On the full paper run (N=800, n_cartels=10): **Hodge grad AUC=0.737**, CIDRE fallback AUC=0.845.

In [ ]:
logger.info(f"Running synthetic Hodge demo: N={N_SYNTH}, n_cartels={N_CARTELS}")

C_synth, journals_synth, gt_synth = generate_synthetic_network(
    N=N_SYNTH, n_fields=N_FIELDS, n_cartels=N_CARTELS,
    cartel_size=CARTEL_SIZE, seed=SYNTH_SEED
)
N_s = C_synth.shape[0]

stacking_ids_synth = {g["openalex_id"] for g in gt_synth}
labels_s = np.array([
    1.0 if journals_synth[i]["id"] in stacking_ids_synth else 0.0
    for i in range(N_s)
])
n_pos_synth = int(labels_s.sum())
logger.info(f"Synthetic n_c={N_CARTELS}: N={N_s}, n_cartel_nodes={n_pos_synth}")

# Run Hodge pipeline on synthetic network
t0 = time.time()
hres_s = hodge_pipeline(C_synth, max_triangles=MAX_TRIANGLES)
elapsed = time.time() - t0
logger.info(f"Hodge pipeline done in {elapsed:.1f}s")

auc_grad_s, ci_grad_s = bootstrap_auc(hres_s["node_grad_residual"], labels_s, B=N_BOOTSTRAP)
auc_curl_s, ci_curl_s = bootstrap_auc(hres_s["node_curl_raw"], labels_s, B=N_BOOTSTRAP)
auc_pres_s, ci_pres_s = bootstrap_auc(hres_s["s_star"], labels_s, B=N_BOOTSTRAP)

print(f"\nSynthetic Hodge Results (N={N_s}, n_cartels={N_CARTELS}):")
print(f"  grad_residual AUC = {auc_grad_s:.3f} CI[{ci_grad_s[0]:.3f},{ci_grad_s[1]:.3f}]")
print(f"  curl_raw AUC      = {auc_curl_s:.3f} CI[{ci_curl_s[0]:.3f},{ci_curl_s[1]:.3f}]")
print(f"  prestige AUC      = {auc_pres_s:.3f} CI[{ci_pres_s[0]:.3f},{ci_pres_s[1]:.3f}]")
print(f"  Energy fractions: {hres_s['energy_fractions']}")
print(f"  Paper (N=800): grad_AUC=0.737, curl_AUC=0.558")

## CIDRE Approximation on Synthetic Network

The real CIDRE package (Sugishita et al. 2021) requires matplotlib 3.1.3, incompatible with Python 3.12. We use an improved approximation: **spectral clustering + Poisson null per block**, which is closer to the dcSBM baseline that CIDRE uses internally.

In [ ]:
def _improved_poisson_cidre(
    A: sparse.csr_matrix,
    N: int,
) -> Tuple[np.ndarray, str, int]:
    """Improved CIDRE approximation: spectral clustering + Poisson null per block."""
    A_sym = ((A + A.T) / 2).toarray()
    try:
        from sklearn.cluster import SpectralClustering
        n_clusters = min(10, N // 10)
        if n_clusters < 2:
            n_clusters = 2
        sc = SpectralClustering(n_clusters=n_clusters, affinity="precomputed",
                                random_state=42, n_init=5)
        labels_sc = sc.fit_predict(A_sym)
    except Exception:
        degrees = A_sym.sum(1)
        labels_sc = np.floor(degrees / (degrees.max() / 10 + 1e-10)).astype(int).clip(0, 9)

    A_arr = A.toarray().astype(float)
    scores = np.zeros(N)
    communities = defaultdict(list)
    for nd, c in enumerate(labels_sc):
        communities[int(c)].append(nd)

    for members in communities.values():
        if len(members) < 2:
            continue
        sub = A_arr[np.ix_(members, members)]
        total = sub.sum()
        if total < 1:
            continue
        row_s = sub.sum(1)
        col_s = sub.sum(0)
        exp = np.outer(row_s, col_s) / (total + 1e-10)
        np.fill_diagonal(exp, 0)
        np.fill_diagonal(sub, 0)
        ratio = sub / (exp + 1e-10)
        for i, nd in enumerate(members):
            scores[nd] = max(scores[nd], float(ratio[i].max()))

    n_groups = int((scores > 2.0).sum())
    return scores, "cidre_poisson_spectral_fallback", n_groups


def run_cidre(
    A: sparse.csr_matrix,
    thresholds: List[float] = None,
) -> Tuple[np.ndarray, str, int]:
    """
    Run real CIDRE package or improved approximation fallback.
    Returns: (score_vector, method_used, n_groups).
    """
    if thresholds is None:
        thresholds = [0.15, 0.05]
    N = A.shape[0]

    try:
        import cidre as cidre_pkg
        for thresh in thresholds:
            groups = cidre_pkg.Cidre(group_membership=None).detect(A, threshold=thresh)
            if len(groups) > 0:
                score = np.zeros(N)
                for g in groups:
                    for nd, p in getattr(g, "donors", {}).items():
                        if 0 <= nd < N:
                            score[nd] = max(score[nd], -np.log(float(p) + 1e-300))
                    for nd, p in getattr(g, "recipients", {}).items():
                        if 0 <= nd < N:
                            score[nd] = max(score[nd], -np.log(float(p) + 1e-300))
                logger.info(f"Real CIDRE: {len(groups)} groups at threshold={thresh}")
                return score, "cidre_package", len(groups)
        return np.zeros(N), "cidre_package_no_groups", 0
    except ImportError:
        logger.warning("cidre package not available, using improved Poisson fallback")
    except Exception as e:
        logger.warning(f"CIDRE package failed: {e}, using fallback")

    return _improved_poisson_cidre(A, N)


logger.info("Running CIDRE on synthetic network...")
cidre_s, method_used_s, n_groups_s = run_cidre(C_synth, thresholds=[0.15, 0.05, 0.01])
auc_cidre_s, ci_cidre_s = bootstrap_auc(cidre_s, labels_s, B=N_BOOTSTRAP)
logger.info(f"CIDRE ({method_used_s}): AUC={auc_cidre_s}, n_groups={n_groups_s}")
print(f"CIDRE ({method_used_s}): AUC={auc_cidre_s:.3f} CI[{ci_cidre_s[0]:.3f},{ci_cidre_s[1]:.3f}]")
print(f"Paper (N=800, n_c=10): CIDRE fallback AUC=0.845")

## Pre-computed Results: Real 231-Journal Network

These results are loaded from `mini_demo_data.json` (pre-computed on the full 231-journal OpenAlex network with 7 JCR-suppressed stacking journals as ground truth).

The real network has 77% curl energy (not gradient-dominant as expected), which means the base network itself has cyclic structure — making detection harder.

In [ ]:
# Display pre-computed AUC results from real 231-journal network
print("=" * 60)
print("PHASE 1: Real 231-Journal Network — Hodge AUC")
print("=" * 60)
auc_table = meta["phase1_auc_table"]
for score_name, results in auc_table.items():
    auc_s = results["auc_stacking_only"]
    ci_s = results["ci_stacking"]
    print(f"  {score_name:30s}: stacking_AUC={auc_s:.3f} CI[{ci_s[0]:.3f},{ci_s[1]:.3f}]")

print()
print("PHASE 2: CIDRE on Real Network")
p2 = meta["phase2_cidre"]
print(f"  Method: {p2['method_used']}")
print(f"  n_groups detected: {p2['n_groups']}")
print(f"  AUC (stacking): {p2['auc_stacking']:.3f}")

print()
print("PHASE 5: Field-Aware Null Model")
p5 = meta["phase5_null_model"]
print(f"  Louvain communities (field proxy): {p5['n_field_communities']}")
print(f"  Spearman ρ(degree_null, field_null): {p5['spearman_rho']:.3f}")
print(f"  Degree-null AUC (stacking): {p5['degree_null_auc_stacking']:.3f}")
print(f"  Field-null AUC (stacking):  {p5['field_null_auc_stacking']:.3f}  ← BEST")
print(f"  Recommendation: {p5['recommendation']}")

print()
print("PHASE 6: Energy Fractions")
p6 = meta["phase6_energy"]
print(f"  Real network:  grad={p6['real_gradient']:.3f}, curl={p6['real_curl']:.3f}")
print(f"  Synth n_c=10:  grad={p6['synth_gradient']:.3f}, curl={p6['synth_curl']:.3f}")
print(f"  Delta curl (synth - real): {p6['delta_curl']:.3f}")

## Journal Scores from Loaded Data

Each journal in the 231-journal network has been scored on 6 prediction fields. Here we display the loaded examples, sorted by field-null z-score (the best-performing predictor).

In [ ]:
# Display journal scores from loaded data
print(f"{'Journal':<35} {'Label':<28} {'Grad':<8} {'Curl':<8} {'FieldZ':<8} {'CIDRE':<8}")
print("-" * 95)

# Sort by field-null z-score
sorted_examples = sorted(examples,
    key=lambda e: float(e['predict_curl_z_field_null']), reverse=True)

for ex in sorted_examples:
    name = ex['metadata_journal_name'][:34]
    label = ex['output']
    grad = float(ex['predict_hodge_grad_residual'])
    curl = float(ex['predict_hodge_curl_raw'])
    fz = float(ex['predict_curl_z_field_null'])
    cidre = float(ex['predict_cidre'])
    print(f"{name:<35} {label:<28} {grad:>7.2f} {curl:>7.1f} {fz:>7.2f} {cidre:>7.2f}")

## Visualization: Results Summary

Summary plots showing:
1. AUC comparison across methods (pre-computed real data + synthetic demo)
2. Hodge energy fractions (real vs synthetic network)
3. Score distributions by journal label in the demo data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Hodge & CIDRE Citation Cartel Detection — Results Summary", fontsize=13, fontweight='bold')

# --- Plot 1: AUC comparison ---
ax1 = axes[0]
methods_real = [
    ('hodge_grad_residual', meta['phase1_auc_table']['hodge_grad_residual']['auc_stacking_only'],
     meta['phase1_auc_table']['hodge_grad_residual']['ci_stacking']),
    ('hodge_curl_raw', meta['phase1_auc_table']['hodge_curl_raw']['auc_stacking_only'],
     meta['phase1_auc_table']['hodge_curl_raw']['ci_stacking']),
    ('hodge_prestige', meta['phase1_auc_table']['hodge_prestige']['auc_stacking_only'],
     meta['phase1_auc_table']['hodge_prestige']['ci_stacking']),
    ('CIDRE_fallback', meta['phase2_cidre']['auc_stacking'], [None, None]),
    ('degree_null_z', meta['phase5_null_model']['degree_null_auc_stacking'], [None, None]),
    ('field_null_z', meta['phase5_null_model']['field_null_auc_stacking'], [None, None]),
]
names = [m[0] for m in methods_real]
aucs = [m[1] for m in methods_real]
colors = ['#e74c3c' if a < 0.5 else '#2ecc71' if a >= 0.65 else '#f39c12' for a in aucs]
bars = ax1.barh(names, aucs, color=colors, alpha=0.8, edgecolor='gray')
ax1.axvline(0.5, color='black', linestyle='--', alpha=0.7, label='Random (0.5)')
for i, (m, auc) in enumerate(zip(methods_real, aucs)):
    ci = m[2]
    if ci[0] is not None:
        ax1.errorbar(auc, i, xerr=[[auc-ci[0]], [ci[1]-auc]], fmt='none', color='black', capsize=4)
    ax1.text(max(auc + 0.01, 0.52), i, f'{auc:.3f}', va='center', fontsize=8)
ax1.set_xlim(0, 0.95)
ax1.set_xlabel('AUC (stacking-only labels, real 231-journal network)')
ax1.set_title('Phase 1+2+5: AUC by Method\n(231-journal real network)', fontsize=10)
ax1.legend(fontsize=8)

# --- Plot 2: Energy fractions ---
ax2 = axes[1]
p6 = meta['phase6_energy']
fracs_real = [p6['real_gradient'], p6['real_curl'], 1 - p6['real_gradient'] - p6['real_curl']]
fracs_synth = [p6['synth_gradient'], p6['synth_curl'], 1 - p6['synth_gradient'] - p6['synth_curl']]
demo_synth_energy = hres_s['energy_fractions']
fracs_demo = [demo_synth_energy['gradient'], demo_synth_energy['curl'], demo_synth_energy['harmonic']]
labels_ef = ['Gradient', 'Curl', 'Harmonic']
ef_colors = ['#3498db', '#e74c3c', '#9b59b6']
x = np.arange(3)
width = 0.25
ax2.bar(x - width, fracs_real, width, label='Real (231 journals)', color=ef_colors, alpha=0.9)
ax2.bar(x, fracs_synth, width, label='Synth paper (N=800, n_c=10)', color=ef_colors, alpha=0.5, hatch='//')
ax2.bar(x + width, fracs_demo, width, label=f'Synth demo (N={N_SYNTH}, n_c={N_CARTELS})',
        color=ef_colors, alpha=0.3, hatch='xx')
ax2.set_xticks(x)
ax2.set_xticklabels(labels_ef)
ax2.set_ylabel('Energy fraction')
ax2.set_title('Phase 6: Hodge Energy Fractions\nReal vs Synthetic', fontsize=10)
ax2.legend(fontsize=7)
ax2.set_ylim(0, 1.05)

# --- Plot 3: Score distribution by label ---
ax3 = axes[2]
label_groups = defaultdict(list)
for ex in examples:
    fz = float(ex['predict_curl_z_field_null'])
    label_groups[ex['output']].append(fz)

label_order = ['suppressed_stacking', 'suppressed_self_citation', 'not_suppressed']
label_colors_plot = {'suppressed_stacking': '#e74c3c', 'suppressed_self_citation': '#f39c12', 'not_suppressed': '#2ecc71'}
label_short = {'suppressed_stacking': 'Stacking (n=7)', 'suppressed_self_citation': 'Self-cit (n=5)', 'not_suppressed': 'Not supp (n=5)'}

positions = []
for i, lbl in enumerate(label_order):
    vals = label_groups.get(lbl, [])
    if vals:
        bp = ax3.boxplot(vals, positions=[i], widths=0.5, patch_artist=True,
                         boxprops=dict(facecolor=label_colors_plot[lbl], alpha=0.7),
                         medianprops=dict(color='black', linewidth=2))
        ax3.scatter([i] * len(vals), vals, color=label_colors_plot[lbl],
                    alpha=0.9, zorder=5, s=30, edgecolors='black', linewidths=0.5)
        positions.append(i)

ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax3.set_xticks(list(range(len(label_order))))
ax3.set_xticklabels([label_short[l] for l in label_order], rotation=12, fontsize=8)
ax3.set_ylabel('Field-Null Curl Z-Score')
ax3.set_title('Phase 5: Field-Null Z-Scores\nby Journal Label (demo data)', fontsize=10)

plt.tight_layout()
plt.savefig('results_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nFigure saved to results_summary.png")

# --- Final summary table ---
print("\n" + "=" * 60)
print("COMBINED AUC SUMMARY (Primary label: stacking_only, n=7)")
print("=" * 60)
combined = meta['combined_auc_primary_label']
for method, auc in combined.items():
    if auc is not None:
        tag = " ← BEST" if auc == max(v for v in combined.values() if v is not None) else ""
        print(f"  {method:<30}: AUC = {auc:.3f}{tag}")

print(f"\nSynthetic demo (N={N_SYNTH}, n_c={N_CARTELS}):")
print(f"  grad_residual AUC = {auc_grad_s:.3f}" if auc_grad_s else "  grad_residual: N/A")
print(f"  curl_raw AUC      = {auc_curl_s:.3f}" if auc_curl_s else "  curl_raw: N/A")
print(f"  CIDRE fallback    = {auc_cidre_s:.3f}" if auc_cidre_s else "  CIDRE: N/A")